In [7]:
!pip install transformers datasets accelerate gradio


In [8]:
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments

In [9]:
import os
os.listdir()

['.config', '.gradio', 'sample_data']

In [16]:
import pandas as pd

df = pd.read_csv("college_dataset.csv")   # change name if different

df = df.dropna()   # remove empty rows
df['text'] = df['text'].astype(str)  # convert everything to string
df.head()

,text,label
0,hi,greeting
1,hello,greeting
2,hey,greeting
3,hello brother,greeting
4,how are you,greeting


In [17]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])

num_labels = len(df['label'].unique())
print("Number of intents:", num_labels)

Number of intents: 10


In [18]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

def tokenize(example):
    return tokenizer(example['text'], truncation=True, padding='max_length')

dataset = dataset.map(tokenize, batched=True)

dataset = dataset.train_test_split(test_size=0.2)

Map:   0%|          | 0/145 [00:00<?, ? examples/s]

In [21]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    logging_dir="./logs",
    logging_steps=10
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [23]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test']
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,1.072287


Step,Training Loss
10,1.072287
20,0.717525
30,0.540912
40,0.366810
50,0.293146
60,0.249288


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=60, training_loss=0.539994748433431, metrics={'train_runtime': 1799.241, 'train_samples_per_second': 0.258, 'train_steps_per_second': 0.033, 'total_flos': 61473642086400.0, 'train_loss': 0.539994748433431, 'epoch': 4.0})

In [24]:
import torch.nn.functional as F

def predict_intent(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)
    confidence, pred = torch.max(probs, dim=1)

    print("Confidence:", confidence.item())  # DEBUG

    if confidence.item() < 0.2:
        return "fallback"

    return le.inverse_transform([pred.item()])[0]

In [25]:
import random

responses = {
    "greeting": [
        "Hey bro 😄 what help do you need?",
        "Hello! How can I help you?",
        "Hey buddy, how are you?",
        "Tell me, what do you need help with? 🙌",
        "Hey! How can I help you, buddy?"
    ],

    "placement_drive": [
        "Placement updates are usually shared in the college placement group.",
        "You can get the latest info from the placement cell 👍",
        "Placements usually start from the third year 👍",
        "Try contacting placement cell members!",
        "The highest package has gone up to 50 LPA 😉",
        "The placement cell is on the 5th floor of the architecture building.",
        "Contact the placement cell for more information 👌"
    ],

    "college_fees": [
        "Fee details are available on the college website 💰",
        "The fee structure varies depending on your branch.",
        "Fees can be paid via cash or online. For more info, contact the admin office.",
        "Fees are usually paid at the start of the semester.",
        "You can pay fees online as well. Check with the admin office ✌️",
        "You can get the fee receipt from the admin building 😊",
        "All fee details are available on the college website 🙌"
    ],

    "exam_form": [
        "Exam forms are filled through the college portal 📅",
        "Make sure not to miss the last date 😅",
        "You will receive a mail when form filling starts 📩",
        "You will get a mail for backlog forms 📩",
        "You can check the last date on the portal 😉",
        "The exam form fee is around 3500 INR 🥲"
    ],

    "result_date": [
        "Results are usually declared 15–20 days after exams 📊",
        "You can check your result on the college portal.",
        "Marks are usually available within 15–20 days.",
        "Open day usually starts the next day after exams 😊"
    ],

    "attendance_criteria": [
        "A minimum of 75% attendance is required 📘",
        "Low attendance may cause issues in exam eligibility 😬",
        "Yes, 75% attendance is compulsory, otherwise you may not be allowed to sit for exams 🥲",
        "If your attendance is low, you may need to complete tasks given by your teacher 🥲",
        "Attendance is calculated based on your lecture attendance 😊"
    ],

    "campus_events": [
        "College fest updates are shared on the notice board or Instagram 🎉",
        "Events happen regularly, check the college page.",
        "Swartarang is usually held in February 🤩",
        "Event registration forms are shared in student groups 🙌",
        "College days usually start in the first week of February 🤩",
        "Innoveda is an IT department event where research papers are presented 🎊"
    ],

    "syllabus": [
        "The syllabus is available on the college website 📚",
        "You can download the official syllabus PDF online 📝",
        "You can find the syllabus on the PCCOE website 📝",
        "The syllabus is available on the college website 📋"
    ],

    "hostel_facilities": [
        "Hostel facilities are decent, and WiFi is available 🏠",
        "Usually, 3 people share one room ✌️",
        "You can find rules in the hostel rule book 📝",
        "Hostel and mess are both compulsory 😉",
        "Facilities like WiFi, laundry, AC/non-AC rooms are available (check website for details) 📝",
        "Hostel allotment is done after admission."
    ],

    "mess": [
        "Mess food is sometimes good, sometimes average 😂",
        "You can also try nearby canteens for better options 🍔",
        "Mess provides breakfast, lunch, evening snacks, and dinner 🤤",
        "Veg food is available in both mess facilities.",
        "Yes, mess is compulsory if you stay in the hostel 👌",
        "The canteen is located on the ground floor of the girls hostel and in the architecture basement 🤤",
        "Non-veg food is available in boys hostel and new girls hostel 😉",
        "Mess charges are included in the fee structure on the college website."
    ],

    "fallback": [
        "I didn’t understand that 😅 please rephrase.",
        "That seems out of scope 🤔",
        "I don’t have exact information on that 😬"
    ]
}

last_intent = None


In [26]:
def chatbot_response(user_input):
    intent = predict_intent(user_input)
    return random.choice(responses.get(intent, responses["fallback"]))

In [27]:
import gradio as gr

def chat(user_input):
    return chatbot_response(user_input)

gr.Interface(fn=chat, inputs="text", outputs="text").launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://65d71ca697605d0fb2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
